# Join crypto prices and Google Trends

This notebook merges the cleaned price dataset and cleaned Google Trends dataset into one daily file for ML.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

PRICES_PATH = "crypto_prices_clean.csv"
TRENDS_PATH = "google_trends_clean.csv"
OUTPUT_PATH = "eth_ml_dataset.csv"

In [2]:
# Load both cleaned datasets
prices = pd.read_csv(PRICES_PATH)
trends = pd.read_csv(TRENDS_PATH)

# Standardize date column names
prices = prices.rename(columns={"Date": "date"})

# Parse dates and clean index order
prices["date"] = pd.to_datetime(prices["date"], errors="coerce")
trends["date"] = pd.to_datetime(trends["date"], errors="coerce")

prices = prices.dropna(subset=["date"]).sort_values("date")
trends = trends.dropna(subset=["date"]).sort_values("date")

# Remove duplicate dates if any
prices = prices.drop_duplicates(subset=["date"], keep="last")
trends = trends.drop_duplicates(subset=["date"], keep="last")

print("Prices shape:", prices.shape)
print("Trends shape:", trends.shape)
prices.head()

Prices shape: (2077, 4)
Trends shape: (3019, 16)


,date,BTC,VIX,ETH
0,2018-01-02,14982.099609,9.77,884.443970
1,2018-01-03,15201.000000,9.15,962.719971
2,2018-01-04,15599.200195,9.22,980.921997
3,2018-01-05,17429.500000,9.22,997.719971
4,2018-01-08,15170.099609,9.52,1148.530029


In [3]:
# Quick date coverage check
print("Prices date range:", prices["date"].min().date(), "->", prices["date"].max().date())
print("Trends date range:", trends["date"].min().date(), "->", trends["date"].max().date())

common_start = max(prices["date"].min(), trends["date"].min())
common_end = min(prices["date"].max(), trends["date"].max())
print("Common overlap:", common_start.date(), "->", common_end.date())

Prices date range: 2018-01-02 -> 2026-04-08
Trends date range: 2018-01-01 -> 2026-04-07
Common overlap: 2018-01-02 -> 2026-04-07


In [4]:
# Merge on date (inner join keeps only dates present in both)
df_ml = prices.merge(trends, on="date", how="inner")

# Keep deterministic order
df_ml = df_ml.sort_values("date").reset_index(drop=True)

# Ensure numeric columns are numeric
for col in df_ml.columns:
    if col != "date":
        df_ml[col] = pd.to_numeric(df_ml[col], errors="coerce")

print("Merged shape:", df_ml.shape)
print("Missing values total:", int(df_ml.isna().sum().sum()))
print("Columns:", df_ml.columns.tolist())
df_ml.head()

Merged shape: (2076, 19)
Missing values total: 0
Columns: ['date', 'BTC', 'VIX', 'ETH', 'bitcoin', 'cryptocurrency', 'ethereum', 'crypto', 'btc', 'eth', 'crypto crash', 'bitcoin crash', 'buy bitcoin', 'sell bitcoin', 'crypto bull run', 'solana', 'altcoin', 'bitcoin price', 'crypto wallet']


,date,BTC,VIX,ETH,bitcoin,cryptocurrency,ethereum,crypto,btc,eth,crypto crash,bitcoin crash,buy bitcoin,sell bitcoin,crypto bull run,solana,altcoin,bitcoin price,crypto wallet
0,2018-01-02,14982.099609,9.77,884.443970,52,10,8,7,11,67,0,3,42,5,0,1,2,49,1
1,2018-01-03,15201.000000,9.15,962.719971,54,12,8,9,12,70,1,3,52,7,0,1,3,46,1
2,2018-01-04,15599.200195,9.22,980.921997,52,14,9,11,13,91,1,3,65,6,0,1,4,42,2
3,2018-01-05,17429.500000,9.22,997.719971,50,13,8,10,12,81,1,3,51,7,0,1,4,41,2
4,2018-01-08,15170.099609,9.52,1148.530029,49,12,11,10,11,88,2,3,42,6,0,1,3,42,1


In [5]:
# Final validation
print("Date range:", df_ml["date"].min().date(), "->", df_ml["date"].max().date())
print("Duplicate dates:", int(df_ml["date"].duplicated().sum()))
print("Rows:", len(df_ml))
print("Columns:", len(df_ml.columns))

df_ml.tail()

Date range: 2018-01-02 -> 2026-04-07
Duplicate dates: 0
Rows: 2076
Columns: 19


,date,BTC,VIX,ETH,bitcoin,cryptocurrency,ethereum,crypto,btc,eth,crypto crash,bitcoin crash,buy bitcoin,sell bitcoin,crypto bull run,solana,altcoin,bitcoin price,crypto wallet
2071,2026-03-31,68233.312500,25.250000,2104.708252,21,1,2,8,8,34,0,0,6,1,0,5,0,16,1
2072,2026-04-01,68078.554688,24.540001,2138.737061,20,1,3,9,8,34,1,0,6,1,0,5,0,16,1
2073,2026-04-02,66888.570312,23.870001,2056.852539,20,1,2,8,8,33,1,0,7,1,0,5,0,15,1
2074,2026-04-06,68859.828125,24.170000,2107.761475,20,1,3,10,8,33,1,1,8,1,0,6,0,16,1
2075,2026-04-07,71940.703125,25.780001,2241.806641,21,1,2,9,8,33,1,1,7,1,0,5,0,14,1


In [6]:
# Save final ML dataset
df_ml.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")

Saved: eth_ml_dataset.csv
